# Project 1: AI vs Human Text Detection

This notebook builds machine learning and deep learning classifiers to predict whether a text sample was written by a human or generated by AI.

**Label meanings:**
- `0` = Human-written text
- `1` = AI-written text

The notebook is organized into the required project sections:
1. Data Exploration & Preprocessing
2. Feature Engineering
3. Model Training & Tuning
4. Evaluation & Comparison


# 1. Data Exploration & Preprocessing

In this section, we load the dataset, inspect its structure, check for missing values, visualize class balance, and clean the raw text so it can be used by machine learning models.


In [ ]:
# Import the main libraries used throughout the project.
# pandas/numpy: data loading and manipulation
# matplotlib/seaborn: visualizations
# re: regular expressions for text cleaning

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Scikit-learn tools for splitting data, feature extraction, model training, and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the Excel dataset that was already uploaded into the Colab environment.
# The file must be in the current Colab working directory.
df = pd.read_excel("../data/train_data with labels.xlsx")

# Display the first five rows to confirm the data loaded correctly.
df.head()


In [ ]:
# Check the size of the dataset and the data types of each column.
# This helps confirm that the dataset has the expected columns: text and label.

print("Dataset Shape:", df.shape)
df.info()

# Check how many examples belong to each class.
# A balanced dataset should have similar counts for labels 0 and 1.
print("Class Counts:")
print(df["label"].value_counts())

# Check for missing values.
# Missing text or labels could cause model training errors.
print("Missing Values:")
print(df.isnull().sum())


In [ ]:
# Visualize the number of human-written and AI-written samples.
# This is important because imbalanced data can bias a classifier toward the majority class.

plt.figure(figsize=(6,4))
sns.countplot(x="label", data=df)
plt.title("Class Distribution: Human vs AI")
plt.xlabel("Label (0 = Human, 1 = AI)")
plt.ylabel("Count")
plt.show()


In [ ]:
# Create a new feature that measures the number of characters in each text sample.
# Text length can reveal patterns between human and AI writing.

df["text_length"] = df["text"].apply(len)

# Show summary statistics such as mean, min, max, and quartiles.
print(df["text_length"].describe())

# Plot the distribution of text lengths.
plt.figure(figsize=(8,5))
sns.histplot(df["text_length"], bins=40, kde=True)
plt.title("Text Length Distribution")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Text cleaning function.
# This prepares raw text for TF-IDF by:
# 1. Converting all text to lowercase
# 2. Removing numbers, punctuation, and special characters
# 3. Removing extra spaces

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply cleaning to every text sample and store it in a new column.
df["clean_text"] = df["text"].apply(clean_text)

# Compare the original text with the cleaned version.
df[["text", "clean_text", "label"]].head()


# 2. Feature Engineering

Machine learning models cannot directly understand raw text, so the text must be converted into numerical features. This project uses TF-IDF features, learned embedding features for deep learning, and linguistic features.


In [ ]:
# Split raw text first. Fit feature extraction only on training text.
train_text, test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.20,
    random_state=42, stratify=df["label"]
)
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2))
X_train = tfidf.fit_transform(train_text)
X_test = tfidf.transform(test_text)


In [ ]:
print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


In [ ]:
# Linguistic features capture writing style instead of word frequency.
# These features can help compare human and AI writing patterns.

import string

# Number of words in each text sample
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

# Average length of words in each text sample
df["avg_word_length"] = df["clean_text"].apply(
    lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
)

# Number of punctuation marks in the original text
df["punctuation_count"] = df["text"].apply(
    lambda x: sum(1 for c in str(x) if c in string.punctuation)
)

# Vocabulary richness: unique words divided by total words
df["vocab_richness"] = df["clean_text"].apply(
    lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0
)

df[["word_count", "avg_word_length", "punctuation_count", "vocab_richness"]].head()


**Word Embedding Explanation**

For the deep learning models, the text is converted into integer sequences using a tokenizer. Then, an `Embedding` layer learns dense vector representations of words during training. These learned embeddings allow LSTM and CNN models to capture semantic patterns and word relationships.


# 3. Model Training & Tuning

This section trains the six required models:

Traditional Machine Learning:
1. Support Vector Machine
2. Decision Tree
3. AdaBoost

Deep Learning:
4. Feedforward Neural Network
5. LSTM
6. CNN for Text


In [ ]:
# Dictionary used to store model accuracies.
# This will later be converted into a comparison table.
results = {}


## Model 1: Support Vector Machine (SVM)

SVM is commonly strong for text classification because TF-IDF creates high-dimensional sparse features. A linear SVM is used because it trains much faster than the full kernel SVM while still performing very well on this dataset.


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Create and train the SVM model.
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

# Predict labels for the test set.
svm_pred = svm_model.predict(X_test)

# Calculate accuracy.
svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", svm_accuracy)
print("Classification Report:")
print(classification_report(y_test, svm_pred))

# Save the result for the final comparison table.
results["SVM"] = svm_accuracy


In [ ]:
# Confusion matrix for SVM.
# The matrix shows correct and incorrect predictions for each class.

cm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Model 2: Decision Tree

A Decision Tree learns a series of rule-based splits. It is easier to interpret than many other models, but it may not perform as well as SVM on high-dimensional TF-IDF features.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# GridSearchCV tests multiple parameter combinations and chooses the best one.
# max_depth controls how deep the tree can grow.
# min_samples_split controls how many samples are required to split a node.

dt_params = {
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best Decision Tree Parameters:")
print(grid_dt.best_params_)


In [ ]:
# Evaluate the best Decision Tree model from GridSearchCV.

best_dt = grid_dt.best_estimator_
dt_pred = best_dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy:", dt_accuracy)
print("Classification Report:")
print(classification_report(y_test, dt_pred))

results["Decision Tree"] = dt_accuracy


In [ ]:
# Confusion matrix for Decision Tree.

cm = confusion_matrix(y_test, dt_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Model 3: AdaBoost

AdaBoost combines many weak learners into a stronger classifier. A smaller fixed parameter setting is used here to reduce training time while still meeting the requirement to train an AdaBoost model.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost can be slow with large grid searches, so this version uses a practical fixed setup.
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)
ada_accuracy = accuracy_score(y_test, ada_pred)

print("AdaBoost Accuracy:", ada_accuracy)
print("Classification Report:")
print(classification_report(y_test, ada_pred))

results["AdaBoost"] = ada_accuracy


In [ ]:
# Confusion matrix for AdaBoost.

cm = confusion_matrix(y_test, ada_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title("AdaBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Model 4: Feedforward Neural Network (FNN)

The FNN uses TF-IDF vectors as input. It has two hidden layers with dropout to reduce overfitting.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Neural networks require dense arrays instead of sparse TF-IDF matrices.
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

# Build the Feedforward Neural Network.
fnn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_dense.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Binary crossentropy is used because this is a binary classification problem.
fnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model.
history = fnn_model.fit(
    X_train_dense,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
# Evaluate the FNN model on the test set.

fnn_loss, fnn_accuracy = fnn_model.evaluate(X_test_dense, y_test)
print("FNN Accuracy:", fnn_accuracy)

# Convert predicted probabilities into class labels.
fnn_pred = (fnn_model.predict(X_test_dense) > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, fnn_pred))

results["FNN"] = fnn_accuracy


## Sequence Preparation for LSTM and CNN

LSTM and CNN models use tokenized sequences instead of TF-IDF vectors. The tokenizer converts words into integer IDs, and padding makes every sequence the same length.


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 500
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_text)
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_text), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_text), maxlen=max_len)
y_train_seq, y_test_seq = y_train, y_test


## Model 5: LSTM

LSTM is a recurrent neural network that reads text as a sequence. It can capture word order and long-range patterns, although it may require more tuning than TF-IDF-based models.


In [ ]:
from tensorflow.keras.layers import Embedding, LSTM

# Build the LSTM model.
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()


In [ ]:
# Train the LSTM model.
# If training is slow, keep epochs low for this project.

lstm_history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=3,
    batch_size=32,
    validation_split=0.20
)


In [ ]:
# Evaluate the LSTM model.

lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test_seq, y_test_seq)
print("LSTM Accuracy:", lstm_accuracy)

lstm_pred = (lstm_model.predict(X_test_seq) > 0.5).astype(int)
print("Classification Report:")
print(classification_report(y_test_seq, lstm_pred))

results["LSTM"] = lstm_accuracy


## Model 6: CNN for Text

A text CNN uses convolution filters to detect useful local word patterns, such as short phrases or n-grams. It is usually faster than LSTM and often performs well for text classification.


In [ ]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

# Build the CNN model for text classification.
cnn_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


In [ ]:
# Train the CNN model.

cnn_history = cnn_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=3,
    batch_size=32,
    validation_split=0.20
)


In [ ]:
# Evaluate the CNN model.

cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test_seq, y_test_seq)
print("CNN Accuracy:", cnn_accuracy)

cnn_pred = (cnn_model.predict(X_test_seq) > 0.5).astype(int)
print("Classification Report:")
print(classification_report(y_test_seq, cnn_pred))

results["CNN"] = cnn_accuracy


# 4. Evaluation & Comparison

This section compares all six models using accuracy and visualizes the results. The strongest models can later be used in the Streamlit web application.


In [ ]:
# Convert the results dictionary into a DataFrame for easier comparison.

results_df = pd.DataFrame(
    list(results.items()),
    columns=["Model", "Accuracy"]
)

results_df = results_df.sort_values(by="Accuracy", ascending=False)
results_df


In [ ]:
# Bar chart comparing model accuracy.

plt.figure(figsize=(9,5))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.title("Model Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.show()


## Written Analysis

The Feedforward Neural Network, SVM, and CNN performed the best overall, each achieving very high accuracy. These results show that TF-IDF-based features and CNN sequence-based features are effective for separating AI-written and human-written text in this dataset.

The Decision Tree performed reasonably well but was weaker than SVM and FNN, likely because a single tree struggles with high-dimensional TF-IDF data. AdaBoost improved over a basic weak learner by combining multiple models. The LSTM may perform poorly without additional tuning, more epochs, or pretrained embeddings, showing that sequence models are not automatically better than traditional machine learning approaches.

Overall, the best model for deployment would be the model with the highest test accuracy, while SVM is also a strong option because it is fast, stable, and effective for text classification.


## Save Models for Streamlit App

The trained models and vectorizers should be saved so that the Streamlit web app can load them later without retraining.


In [ ]:
%who

In [ ]:
from pathlib import Path
import joblib

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)
joblib.dump(svm_model, model_dir / "svm_model.pkl")
joblib.dump(best_dt, model_dir / "decision_tree_model.pkl")
joblib.dump(ada_model, model_dir / "adaboost_model.pkl")
joblib.dump(tfidf, model_dir / "tfidf_vectorizer.pkl")
fnn_model.save(model_dir / "fnn_model.keras")
lstm_model.save(model_dir / "lstm_model.keras")
cnn_model.save(model_dir / "cnn_model.keras")
